# Module F: Lap Visualizer & Circuit Map

**F1 Race Intelligence Project**

This notebook creates GPS-based circuit visualizations using FastF1 telemetry data.
By plotting car position (X, Y coordinates) and coloring by speed or speed delta,
we can see exactly where on the circuit each driver gains or loses time.

**Visualizations:**
1. Circuit GPS speed map (single driver, colored by speed)
2. Speed delta map (two drivers, colored by who is faster at each point)
3. Brake zone identification
4. Throttle application map
5. Multi-driver lap comparison on the circuit outline

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.viz import circuit_speed_map, f1_layout, F1_RED, F1_WHITE, F1_BG, TEAM_COLORS

TELEMETRY_DIR = Path('../data/raw/telemetry')

# Configuration — using Bahrain GP where both drivers completed the full race
GP_NAME = '2024_bahrain_grand_prix'
DRIVER_A = 'VER'
DRIVER_B = 'NOR'
CIRCUIT_LENGTH_M = 5412  # Bahrain International Circuit length in meters

print(f'Circuit visualization: {GP_NAME}')
print(f'Drivers: {DRIVER_A} vs {DRIVER_B}')

## 1. Load Telemetry Data

We load the raw telemetry parquet files which contain GPS coordinates (X, Y),
speed, throttle percentage, and brake status at approximately 240 Hz.

In [ ]:
# Load telemetry for both drivers
tel_a_path = TELEMETRY_DIR / f'{GP_NAME}_R_{DRIVER_A}_tel.parquet'
tel_b_path = TELEMETRY_DIR / f'{GP_NAME}_R_{DRIVER_B}_tel.parquet'

tel_a_full = pd.read_parquet(tel_a_path)
tel_b_full = pd.read_parquet(tel_b_path)

print(f'{DRIVER_A} full telemetry: {tel_a_full.shape[0]:,} data points')
print(f'{DRIVER_B} full telemetry: {tel_b_full.shape[0]:,} data points')
print(f'\nColumns: {list(tel_a_full.columns)}')

# Extract a single representative lap from cumulative telemetry
# Distance is cumulative across the full race, so we split by circuit length
def extract_fastest_lap(tel_full, circuit_length_m, margin=0.15):
    """Extract the fastest single lap from cumulative telemetry."""
    laps = []
    dist = tel_full['Distance'].values
    for start_idx in range(len(dist)):
        if dist[start_idx] % circuit_length_m < circuit_length_m * 0.05:
            # Find end of this lap
            target_dist = dist[start_idx] + circuit_length_m
            end_indices = np.where(
                (dist >= target_dist * (1 - margin)) &
                (dist <= target_dist * (1 + margin))
            )[0]
            end_indices = end_indices[end_indices > start_idx]
            if len(end_indices) > 0:
                end_idx = end_indices[0]
                lap_data = tel_full.iloc[start_idx:end_idx].copy()
                if len(lap_data) > 100:
                    lap_dist = dist[end_idx] - dist[start_idx]
                    avg_speed = lap_data['Speed'].mean()
                    laps.append((start_idx, end_idx, lap_dist, avg_speed, lap_data))

    if not laps:
        # Fallback: pick a chunk from the middle of the race
        mid = len(tel_full) // 2
        chunk = tel_full.iloc[mid:mid + len(tel_full) // 57].copy()
        chunk['Distance'] = chunk['Distance'] - chunk['Distance'].iloc[0]
        return chunk

    # Pick the lap with highest average speed (likely clean, no traffic)
    best = max(laps, key=lambda x: x[3])
    lap = best[4].copy()
    lap['Distance'] = lap['Distance'] - lap['Distance'].iloc[0]
    return lap

tel_a = extract_fastest_lap(tel_a_full, CIRCUIT_LENGTH_M)
tel_b = extract_fastest_lap(tel_b_full, CIRCUIT_LENGTH_M)

print(f'\n{DRIVER_A} single lap: {len(tel_a):,} points')
print(f'{DRIVER_B} single lap: {len(tel_b):,} points')
print(f'\nSpeed range ({DRIVER_A}): {tel_a["Speed"].min():.0f} - {tel_a["Speed"].max():.0f} km/h')

# Check for GPS data
has_gps = 'X' in tel_a.columns and 'Y' in tel_a.columns
print(f'\nGPS data available: {has_gps}')

if has_gps:
    print(f'X range: {tel_a["X"].min():.0f} to {tel_a["X"].max():.0f}')
    print(f'Y range: {tel_a["Y"].min():.0f} to {tel_a["Y"].max():.0f}')

display(tel_a.head(5))

## 2. Circuit GPS Speed Map

The circuit outline rendered from GPS coordinates, with each point colored by the car's speed.
This reveals the circuit's character: braking zones (blue/slow), straights (red/fast),
and corner speeds.

In [ ]:
if has_gps:
    # Single driver speed map
    fig = go.Figure(go.Scatter(
        x=tel_a['X'], y=tel_a['Y'],
        mode='markers',
        marker=dict(
            size=3,
            color=tel_a['Speed'],
            colorscale=[
                [0.0, '#0000FF'],   # slow = blue
                [0.3, '#00FFFF'],   # medium-slow = cyan
                [0.5, '#00FF00'],   # medium = green
                [0.7, '#FFFF00'],   # medium-fast = yellow
                [1.0, '#FF0000'],   # fast = red
            ],
            colorbar=dict(title='Speed (km/h)'),
        ),
        hovertemplate='Speed: %{marker.color:.0f} km/h<extra></extra>'
    ))
    fig = f1_layout(fig, f'Circuit Speed Map: {DRIVER_A} — {GP_NAME.replace("_", " ").title()}', height=600)
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False, scaleanchor='x')
    fig.show()
else:
    print('No GPS data available. Speed map requires X, Y coordinates in telemetry.')

## 3. Speed Delta Map: VER vs NOR

The most insightful visualization: the circuit colored by the speed difference between
two drivers. Red sections mean Driver A is faster, blue sections mean Driver B is faster.
This immediately shows where each driver has an advantage on the track.

In [ ]:
if has_gps:
    # Align telemetry by resampling to common distance points
    # Use the shorter telemetry as reference length
    min_len = min(len(tel_a), len(tel_b))
    
    # If Distance column exists, use it for alignment
    if 'Distance' in tel_a.columns and 'Distance' in tel_b.columns:
        max_dist = min(tel_a['Distance'].max(), tel_b['Distance'].max())
        ref_dist = np.linspace(0, max_dist, num=2000)
        
        aligned_a = pd.DataFrame({
            'X': np.interp(ref_dist, tel_a['Distance'].values, tel_a['X'].values),
            'Y': np.interp(ref_dist, tel_a['Distance'].values, tel_a['Y'].values),
            'Speed': np.interp(ref_dist, tel_a['Distance'].values, tel_a['Speed'].values),
        })
        aligned_b = pd.DataFrame({
            'Speed': np.interp(ref_dist, tel_b['Distance'].values, tel_b['Speed'].values),
        })
    else:
        # Fallback: use index-based alignment
        step = max(1, min_len // 2000)
        aligned_a = tel_a.iloc[::step].head(2000).reset_index(drop=True)
        aligned_b = tel_b.iloc[::step].head(2000).reset_index(drop=True)
    
    # Use viz.py circuit_speed_map function
    fig = circuit_speed_map(aligned_a, aligned_b, DRIVER_A, DRIVER_B)
    fig.update_layout(
        title_text=f'Speed Delta Map: {DRIVER_A} vs {DRIVER_B} — {GP_NAME.replace("_", " ").title()}'
    )
    fig.show()
else:
    print('GPS data required for speed delta map')

## 4. Brake Zone Map

Highlighting brake zones on the circuit: the points where drivers hit the brakes.
Later braking points indicate a more aggressive driving style and better braking stability.

In [ ]:
if has_gps and 'Brake' in tel_a.columns:
    # Map brake zones
    fig = go.Figure()
    
    # Circuit outline (no brake)
    no_brake = tel_a[tel_a['Brake'] == False] if tel_a['Brake'].dtype == bool else tel_a[tel_a['Brake'] == 0]
    braking = tel_a[tel_a['Brake'] == True] if tel_a['Brake'].dtype == bool else tel_a[tel_a['Brake'] > 0]
    
    fig.add_trace(go.Scatter(
        x=no_brake['X'], y=no_brake['Y'],
        mode='markers',
        marker=dict(size=2, color='#333333'),
        name='No Brake',
        hoverinfo='skip'
    ))
    fig.add_trace(go.Scatter(
        x=braking['X'], y=braking['Y'],
        mode='markers',
        marker=dict(
            size=4,
            color=F1_RED,
            opacity=0.8
        ),
        name='Braking Zone',
        hovertemplate='Speed: %{text} km/h<extra>Braking</extra>',
        text=braking['Speed'].round(0)
    ))
    
    fig = f1_layout(fig, f'Brake Zone Map: {DRIVER_A} — {GP_NAME.replace("_", " ").title()}', height=600)
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False, scaleanchor='x')
    fig.show()
    
    brake_pct = len(braking) / len(tel_a) * 100
    print(f'Braking: {brake_pct:.1f}% of the lap')
    print(f'Brake zones: {len(braking):,} data points (out of {len(tel_a):,} total)')
else:
    print('Brake data or GPS data not available')

## 5. Throttle Application Map

Similar to the brake map, but showing throttle application percentage.
Full throttle sections appear in bright green, while partial throttle
(traction-limited corners) appear in yellow/orange.

In [ ]:
if has_gps and 'Throttle' in tel_a.columns:
    fig = go.Figure(go.Scatter(
        x=tel_a['X'], y=tel_a['Y'],
        mode='markers',
        marker=dict(
            size=3,
            color=tel_a['Throttle'],
            colorscale=[
                [0.0, '#FF0000'],   # 0% throttle = red
                [0.3, '#FF8800'],   # partial = orange
                [0.6, '#FFFF00'],   # more = yellow
                [1.0, '#00FF00'],   # full throttle = green
            ],
            cmin=0, cmax=100,
            colorbar=dict(title='Throttle %'),
        ),
        hovertemplate='Throttle: %{marker.color:.0f}%<extra></extra>'
    ))
    fig = f1_layout(fig, f'Throttle Map: {DRIVER_A} — {GP_NAME.replace("_", " ").title()}', height=600)
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False, scaleanchor='x')
    fig.show()
    
    full_throttle_pct = (tel_a['Throttle'] >= 95).mean() * 100
    print(f'Full throttle (>= 95%): {full_throttle_pct:.1f}% of the lap')
else:
    print('Throttle or GPS data not available')

## 6. Driver Comparison Overlay

Overlaying both drivers' GPS traces to see if they take different racing lines
through the corners.

In [ ]:
if has_gps:
    # Subsample for cleaner visualization
    step = max(1, len(tel_a) // 3000)
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=tel_a['X'].iloc[::step], y=tel_a['Y'].iloc[::step],
        mode='markers',
        marker=dict(size=2, color=TEAM_COLORS['red_bull'], opacity=0.6),
        name=DRIVER_A
    ))
    fig.add_trace(go.Scatter(
        x=tel_b['X'].iloc[::step], y=tel_b['Y'].iloc[::step],
        mode='markers',
        marker=dict(size=2, color=TEAM_COLORS['mclaren'], opacity=0.6),
        name=DRIVER_B
    ))
    fig = f1_layout(fig, f'Racing Line Overlay: {DRIVER_A} vs {DRIVER_B}', height=600)
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False, scaleanchor='x')
    fig.show()

## 7. Speed Profile Along the Lap

A complementary view: speed as a function of lap distance, with
annotations marking the key braking and acceleration zones.

In [ ]:
if 'Distance' in tel_a.columns:
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                        subplot_titles=('Speed', 'Throttle', 'Brake'),
                        row_heights=[0.5, 0.25, 0.25])
    
    # Speed
    fig.add_trace(go.Scatter(
        x=tel_a['Distance'], y=tel_a['Speed'],
        name=f'{DRIVER_A} Speed',
        line=dict(color=TEAM_COLORS['red_bull'], width=1.5)
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=tel_b['Distance'], y=tel_b['Speed'],
        name=f'{DRIVER_B} Speed',
        line=dict(color=TEAM_COLORS['mclaren'], width=1.5)
    ), row=1, col=1)
    
    # Throttle
    if 'Throttle' in tel_a.columns:
        fig.add_trace(go.Scatter(
            x=tel_a['Distance'], y=tel_a['Throttle'],
            name=f'{DRIVER_A} Throttle',
            line=dict(color=TEAM_COLORS['red_bull'], width=1),
            showlegend=False
        ), row=2, col=1)
        fig.add_trace(go.Scatter(
            x=tel_b['Distance'], y=tel_b['Throttle'],
            name=f'{DRIVER_B} Throttle',
            line=dict(color=TEAM_COLORS['mclaren'], width=1),
            showlegend=False
        ), row=2, col=1)
    
    # Brake
    if 'Brake' in tel_a.columns:
        brake_a = tel_a['Brake'].astype(float)
        brake_b = tel_b['Brake'].astype(float)
        fig.add_trace(go.Scatter(
            x=tel_a['Distance'], y=brake_a,
            name=f'{DRIVER_A} Brake',
            fill='tozeroy',
            line=dict(color=TEAM_COLORS['red_bull'], width=1),
            fillcolor='rgba(54, 113, 198, 0.3)',
            showlegend=False
        ), row=3, col=1)
        fig.add_trace(go.Scatter(
            x=tel_b['Distance'], y=brake_b,
            name=f'{DRIVER_B} Brake',
            fill='tozeroy',
            line=dict(color=TEAM_COLORS['mclaren'], width=1),
            fillcolor='rgba(255, 128, 0, 0.3)',
            showlegend=False
        ), row=3, col=1)
    
    fig = f1_layout(fig, f'Full Lap Profile: {DRIVER_A} vs {DRIVER_B}', height=700)
    fig.update_xaxes(title_text='Distance (m)', row=3)
    fig.update_yaxes(title_text='km/h', row=1)
    fig.update_yaxes(title_text='%', row=2)
    fig.update_yaxes(title_text='On/Off', row=3)
    fig.show()
else:
    print('Distance column not available for lap profile visualization')

## 8. Circuit Characteristics Summary

Extracting key circuit characteristics from the telemetry data.

In [ ]:
# Circuit characteristics from telemetry
print(f'=== Circuit Characteristics: {GP_NAME.replace("_", " ").title()} ===')
print(f'\n{DRIVER_A} Telemetry Summary:')
print(f'  Top speed:        {tel_a["Speed"].max():.0f} km/h')
print(f'  Min speed:        {tel_a["Speed"].min():.0f} km/h')
print(f'  Average speed:    {tel_a["Speed"].mean():.0f} km/h')

if 'Throttle' in tel_a.columns:
    full_throttle = (tel_a['Throttle'] >= 95).mean() * 100
    zero_throttle = (tel_a['Throttle'] <= 5).mean() * 100
    print(f'  Full throttle:    {full_throttle:.1f}%')
    print(f'  Off throttle:     {zero_throttle:.1f}%')

if 'Brake' in tel_a.columns:
    brake_pct = tel_a['Brake'].astype(bool).mean() * 100
    print(f'  Braking:          {brake_pct:.1f}%')

if 'nGear' in tel_a.columns:
    print(f'  Gear changes:     ~{(tel_a["nGear"].diff().abs() > 0).sum()} per lap')
    print(f'  Max gear:         {tel_a["nGear"].max()}')

if has_gps:
    # Approximate lap length from GPS coordinates
    dx = tel_a['X'].diff()
    dy = tel_a['Y'].diff()
    dist = np.sqrt(dx**2 + dy**2).sum()
    print(f'  Approx lap length: {dist:.0f} (arbitrary units)')

# Speed zones breakdown
print(f'\nSpeed Zones ({DRIVER_A}):')
zones = [
    ('< 100 km/h (slow corners)', tel_a['Speed'] < 100),
    ('100-200 km/h (medium)', (tel_a['Speed'] >= 100) & (tel_a['Speed'] < 200)),
    ('200-300 km/h (fast)', (tel_a['Speed'] >= 200) & (tel_a['Speed'] < 300)),
    ('> 300 km/h (straight)', tel_a['Speed'] >= 300),
]
for label, mask in zones:
    pct = mask.mean() * 100
    print(f'  {label}: {pct:.1f}%')

## 9. Available Telemetry Files

For reference, here are all the telemetry files available for further analysis.

In [ ]:
# List all available telemetry for circuit map visualization
tel_files = sorted(TELEMETRY_DIR.glob('*_tel.parquet'))

# Group by GP
gp_drivers = {}
for f in tel_files:
    parts = f.stem.split('_R_')
    if len(parts) == 2:
        gp = parts[0]
        driver = parts[1].replace('_tel', '')
        if gp not in gp_drivers:
            gp_drivers[gp] = []
        gp_drivers[gp].append(driver)

print(f'Grand Prix events with telemetry: {len(gp_drivers)}')
for gp, drivers in sorted(gp_drivers.items()):
    driver_list = ", ".join(sorted(drivers)[:5])
    print(f'  {gp}: {len(drivers)} drivers ({driver_list}...)')

## Key Findings

1. **GPS speed maps** provide an intuitive overview of circuit character: long straights, heavy braking zones, and technical sections are immediately visible.
2. **Speed delta maps** are the most powerful diagnostic tool, revealing exactly where on the circuit one driver gains over another. This directly informs setup decisions.
3. **Brake zone analysis** shows driving style differences: late brakers are visible as red zones that start further down the track.
4. **Throttle maps** reveal traction-limited sections where partial throttle application (and therefore mechanical grip) determines lap time.
5. These visualizations combine with the structured data analysis in Modules A-E to form a complete picture of F1 performance.

---
*This completes the F1 Race Intelligence analytical notebook series.*

In [ ]:
print('Module F complete.')